In [1]:
# Loading data

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../pdfs/textonly.pdf")
text_data = loader.load()
text_data

/tmp/ipykernel_816793/838093174.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/administrator/agents/pdfAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'textonly', 'source': '../pdfs/textonly.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='If  you  asked  me  about  AI  a  year  ago,  I  would  have  told  it  is  all  hype.  Can’t  do  anything  real.  \nYeah,\n \nthat\n \nwas\n \nnaive.\n \nBut,\n \nin\n \nmy\n \ndefense,\n \nI\n \ndid\n \nplay\n \nwith\n \nLLMs\n \nand\n \nthe\n \nresults\n \nwere…\n \nuninspiring.\n \nTried\n \nwriting\n \ncode\n \nand\n \nit\n \nalways\n \nfailed.\n \nTested\n \nhow\n \nwell\n \nit\n \ncould\n \nwrite\n \nstories\n \nand\n \nit\n \nflopped.\n \nSo\n \nbad,\n \nbut\n \nthings\n \nhave\n \nchanged.\n \nFirst,\n \nLLMs\n \nand\n \ntooling\n \naround\n \nthem\n \ngot\n \nway\n \nbetter.\n \nSecond,\n \nI\n \nlearned\n \nhow\n \nto\n \neffectively\n \nuse\n \nLLMs\n \nand\n \nusing\n \nLLMs\n \neffectively\n \nis\n \na\n \nskill\n \nwe\n \nall\n \nneed\n \nin\n \n

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap = 50
)

In [3]:
splitted_text = text_splitter.split_documents(text_data)
splitted_text

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'textonly', 'source': '../pdfs/textonly.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='If  you  asked  me  about  AI  a  year  ago,  I  would  have  told  it  is  all  hype.  Can’t  do  anything  real.  \nYeah,\n \nthat\n \nwas\n \nnaive.\n \nBut,\n \nin\n \nmy\n \ndefense,\n \nI\n \ndid\n \nplay\n \nwith\n \nLLMs\n \nand\n \nthe\n \nresults\n \nwere…\n \nuninspiring.\n \nTried\n \nwriting\n \ncode\n \nand\n \nit\n \nalways\n \nfailed.\n \nTested\n \nhow\n \nwell\n \nit\n \ncould\n \nwrite\n \nstories\n \nand\n \nit\n \nflopped.\n \nSo\n \nbad,\n \nbut\n \nthings\n \nhave\n \nchanged.\n \nFirst,\n \nLLMs\n \nand\n \ntooling\n \naround\n \nthem\n \ngot\n \nway\n \nbetter.'),
 Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'textonly', 'source': '../pdfs/textonly.pdf', 'total_page

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_API_KEY") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 

In [5]:
# Creating embeddings

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splitted_text,
    embedding=embedding
)

In [7]:
retriver = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
retriver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7eab6e341fd0>, search_kwargs={'k': 3})

In [8]:
retriver.invoke(
    input="have changed. First, LLMs and tooling around them got"
)[0].page_content

'tooling\n \naround\n \nthem\n \ngot\n \nway\n \nbetter.\n \nSecond,\n \nI\n \nlearned\n \nhow\n \nto\n \neffectively\n \nuse\n \nLLMs\n \nand\n \nusing\n \nLLMs\n \neffectively\n \nis\n \na\n \nskill\n \nwe\n \nall\n \nneed\n \nin\n \n2026.\n  Let’s  get  one  thing  out  of  the  way.  I  will  never  put  my  name  on  an  article,  story,  course,  or  \nanything\n \nelse\n \nwritten\n \nby\n \nAI\n \nundisclosed.\n \nThis\n \narticle\n \nis\n \n100%\n \nhand-written\n \nby\n \nme,\n \nbut\n \nthe\n \nimage\n \nis\n \nfrom\n \nNanoBanana.'

In [9]:
# llm model

from langchain_groq.chat_models import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",    
    # reasoning_effort= "parsed"  # disable thinking
    reasoning_format="hidden",
)

In [10]:
llm_answer = llm.invoke(input="what is 2+2")
llm_answer.content

'The sum of 2 and 2 is **4**. \n\nIn standard arithmetic (base 10), $2 + 2 = 4$. This applies universally unless specified otherwise (e.g., in different number systems or contextual riddles). \n\n**Answer:** 4.'

In [11]:
messages = [
    (
        "system",
        "You are good at maths and doesnot like to be give long answers. Give the precise one word answer.",
    ),
    ("human", "What is radius of earth"),
]
ai_msg = llm.invoke(messages)
ai_msg.content

'6371 km'

In [24]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}"),
    ("human",
     """
     Context:
     {context}

     Question:
     {question}
     """)
])

In [25]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {
        "context": retriver,
        "question": RunnablePassthrough(),
        "persona": lambda _: "context based assistant and answer only by referring to the context"
    }
    | prompt
    | llm
)

In [26]:
response = chain.invoke("what framework did 10x magic?")
print(response.content)

The document mentions that the author built a website using a modified version of **Spec Kitty**, which was particularly useful early in the project. While "Magic 10X" is referenced as a heading, the specific framework highlighted in the context is **Spec Kitty**. Additionally, **Open Spec** and **BMAD** are noted as popular frameworks for working with LLMs. 

Answer:  
The framework referenced in the context is **Spec Kitty**, a modified version of which was used by the author. Other popular frameworks mentioned include **Open Spec** and **BMAD**.
